<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [1]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Mounted at /content/drive


## 0.2 — Chargement + nettoyage → `inspections`

Une ligne par pont × année (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable).

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go de CSV : compte quelques minutes.

In [2]:
%%sql
-- Chargement + nettoyage en une seule passe : on lit directement les CSV bruts
-- et on ne garde que les colonnes utiles, déjà typées.
CREATE OR REPLACE TABLE inspections AS
WITH raw AS (
    SELECT
        CAST(regexp_extract(filename, 'nbi_(20\d{2})', 1) AS INTEGER) AS year,
        *
    FROM read_csv(
        [   -- 2016 à 2025 : nbi_2015.txt n'est pas un vrai fichier NBI (125 lignes)
            '/content/drive/MyDrive/nbi/raw/nbi_201[6-9].txt',
            '/content/drive/MyDrive/nbi/raw/nbi_202[0-9].txt'
        ],
        encoding = 'latin-1',
        delim = ',',
        quote = '''',            -- le qualificateur de texte NBI est l'apostrophe
        header = true,
        all_varchar = true,      -- tout en texte, on type nous-mêmes ci-dessous
        union_by_name = true,
        filename = true
    )
    WHERE RECORD_TYPE_005A = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
),
typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(YEAR_BUILT_027         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(YEAR_RECONSTRUCTED_106 AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(ADT_029               AS INTEGER) AS adt,
        TRY_CAST(PERCENT_ADT_TRUCK_109 AS INTEGER) AS pct_truck,
        TRY_CAST(STRUCTURE_LEN_MT_049  AS DOUBLE)  AS length_m,
        TRY_CAST(DECK_WIDTH_MT_052     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(DECK_COND_058           AS TINYINT) AS deck,
        TRY_CAST(SUPERSTRUCTURE_COND_059 AS TINYINT) AS super,
        TRY_CAST(SUBSTRUCTURE_COND_060   AS TINYINT) AS sub,
        TRY_CAST(CULVERT_COND_062        AS TINYINT) AS culvert
    FROM raw
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

Running query in 'DuckDBPyConnection'

InvalidInputException: Invalid Input Error: Error when sniffing file "".
It was not possible to automatically detect the CSV Parsing dialect/types
The search space used was:
Delimiter Candidates: ','
Quote/Escape Candidates: [''','(no escape)'],[''','''],[''','"'],[''','\']
Comment Candidates: '\0', '#'
Encoding: latin-1
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Delimiter is set to ','. Consider unsetting it.
* Quote is set to '''. Consider unsetting it.
* Set escape (e.g., escape='"')
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable ignore errors (ignore_errors=true) to ignore potential errors
* Enable null padding (null_padding=true) to pad missing columns with NULL values
* Check you are using the correct file compression, otherwise set

In [ ]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

In [ ]:
%%sql
SELECT * FROM inspections LIMIT 5;

---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [ ]:
%%sql
-- Q1 — nombre de ponts par année


In [ ]:
%%sql
-- Q2 — nombre de ponts distincts


In [ ]:
%%sql
-- Q3 — distribution de min_rating par année


In [ ]:
%%sql
-- Q4 — part de ponceaux
